# TIDE 2.0 Evaluation

Span-level and token-level evaluation of de-identification performance.

**Required inputs:**
- `gold_standard/` — Gold standard JSON files (one per note)
- `pipeline_output/ml_spans.parquet` — Model predictions
- `pipeline_output/texts.parquet` — Original texts (for token-level eval)

In [ ]:
import json
from pathlib import Path

import pandas as pd
from evaluation_config import ABLATION_CONFIGS
from evaluation_config import RECOGNIZER_CATEGORIES
from evaluation_config import UNIFIED_LABEL_MAP
from evaluation_config import get_recognizer_category
from span_eval import run_span_evaluation
from token_eval import run_token_evaluation

from tide2.utils.span_metrics import map_labels

## Configuration

In [ ]:
gold_dir = Path("../sample_data/gold_standard")
output_dir = Path("../sample_data/pipeline_output")
overlap_threshold = 0.8
tokenizer_name = None  # Set to a HuggingFace tokenizer for subword eval; None = char-level

## Load Gold Standard

In [ ]:
rows = []
for jf in sorted(gold_dir.glob("*.json")):
    text_id = jf.stem
    with jf.open() as f:
        spans = json.load(f)
    for s in spans:
        rows.append(
            {
                "text_id": text_id,
                "span_start": s["span_start"],
                "span_end": s["span_end"],
                "span_tag": s["span_tag"],
            }
        )

df_gold = pd.DataFrame(rows, columns=["text_id", "span_start", "span_end", "span_tag"])
df_gold["text_id"] = df_gold["text_id"].astype(str)
df_gold = df_gold.sort_values(["text_id", "span_start"]).reset_index(drop=True)

print(f"Gold spans: {len(df_gold)} ({df_gold['text_id'].nunique()} texts)")
df_gold["span_tag"].value_counts()

## Load ML Predictions

In [ ]:
df_ml = pd.read_parquet(output_dir / "ml_spans.parquet")
if "note_id" in df_ml.columns:
    df_ml = df_ml.rename(columns={"note_id": "text_id"})
df_ml["text_id"] = df_ml["text_id"].astype(str)
if "recognizer_name" in df_ml.columns:
    df_ml["category"] = df_ml["recognizer_name"].apply(lambda x: get_recognizer_category(x, RECOGNIZER_CATEGORIES))
else:
    df_ml["category"] = None

print(f"ML spans: {len(df_ml)} ({df_ml['text_id'].nunique()} texts)")
df_ml["span_tag"].value_counts()

## Apply Unified Label Mapping

In [ ]:
gold_before = len(df_gold)
df_gold = map_labels(df_gold, UNIFIED_LABEL_MAP, column="span_tag", drop_unmapped=True)
ml_before = len(df_ml)
df_ml = map_labels(df_ml, UNIFIED_LABEL_MAP, column="span_tag", drop_unmapped=True)

print(f"Gold spans: {gold_before} -> {len(df_gold)}  (dropped {gold_before - len(df_gold)})")
print(f"ML spans:   {ml_before} -> {len(df_ml)}  (dropped {ml_before - len(df_ml)})")
print(f"\nGold label counts:\n{df_gold['span_tag'].value_counts().to_string()}")
print(f"\nML label counts:\n{df_ml['span_tag'].value_counts().to_string()}")

# Prepare DataFrames for span_metrics (expects note_id column)
df_gold_eval = df_gold.rename(columns={"text_id": "note_id"})
df_ml_eval = df_ml.rename(columns={"text_id": "note_id"})
overlap = set(df_gold_eval["note_id"]) & set(df_ml_eval["note_id"])
print(f"\nOverlapping text IDs: {len(overlap)}")

ablation_configs = list(ABLATION_CONFIGS)

## Span-Level Evaluation

In [ ]:
span_out = output_dir / "span_level_evaluation"

span_df = run_span_evaluation(
    df_gold=df_gold_eval,
    df_ml=df_ml_eval,
    ablation_configs=ablation_configs,
    output_dir=span_out,
    overlap_threshold=overlap_threshold,
)

if not span_df.empty:
    display(span_df[["config", "num_spans", "micro_precision", "micro_recall", "micro_f1", "macro_f1"]])

    # Per-label breakdown for each config
    for config_name, _ in ablation_configs:
        csv_path = span_out / config_name / "per_label_results.csv"
        if csv_path.exists():
            print(f"\n--- {config_name} per-label ---")
            display(pd.read_csv(csv_path))

## Token-Level Evaluation

In [ ]:
token_out = output_dir / "token_level_evaluation"

df_texts = pd.read_parquet(output_dir / "texts.parquet")
if "note_id" in df_texts.columns:
    df_texts = df_texts.rename(columns={"note_id": "text_id"})
df_texts["text_id"] = df_texts["text_id"].astype(str)
print(f"Loaded texts: {len(df_texts)}")

token_df = run_token_evaluation(
    df_gold=df_gold,
    df_ml=df_ml,
    df_texts=df_texts,
    ablation_configs=ablation_configs,
    output_dir=token_out,
    tokenizer_name=tokenizer_name,
)

if not token_df.empty:
    display(token_df[["config", "num_spans", "micro_precision", "micro_recall", "micro_f1", "macro_f1"]])

    # Per-label breakdown for each config
    for config_name, _ in ablation_configs:
        json_path = token_out / config_name / "token_metrics.json"
        if json_path.exists():
            with json_path.open() as f:
                metrics = json.load(f)["metrics"]
            label_rows = [
                {"label": label, **vals}
                for label, vals in metrics.items()
                if label not in ("O", "micro_avg", "macro_avg")
            ]
            if label_rows:
                print(f"\n--- {config_name} per-label ---")
                display(pd.DataFrame(label_rows).sort_values("support", ascending=False).reset_index(drop=True))